[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/02_mathematics_for_ml/08_statistics.ipynb)

# Bài 08 — Thống kê cho Machine Learning

<img src="images/Statistical/statistical-analysis-hero.jpg" style="width:40%;height:40%;">

Chúng ta dùng Python để khám phá các số liệu thống kê, giúp hiểu hơn về dữ liệu đang có. Bài này đi từ các thước đo đơn giản như trung bình (Mean), trung vị (Median), Mode, đến phương sai (Variance), độ lệch chuẩn (Standard Deviation) và hệ số tương quan (Correlation Coefficient). Phần bổ sung trình bày các công cụ thống kê để **đánh giá model một cách đáng tin cậy**: tổng thể vs mẫu, khoảng tin cậy, bootstrap, kiểm định giả thuyết.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/02_mathematics_for_ml

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Mean — Trung bình

Giá trị trung bình là tổng của dãy số chia cho độ dài của dãy:

<img src="images/Statistical/Arithmetic-Mean-Formula.jpg" style="width:35%;height:35%;">

**Ví dụ:** số tiền quyên góp (USD) của một tổ chức từ thiện trong 12 ngày qua: $100, 60, 70, 900, 100, 200, 500, 500, 503, 600, 1000, 1200$.

In [ ]:
def calculate_mean(numbers):
    s = sum(numbers)      # tổng dãy số
    N = len(numbers)      # độ dài dãy số
    return s / N


donations = [100, 60, 70, 900, 100, 200, 500, 500, 503, 600, 1000, 1200]
print('Trung bình số tiền quyên góp là:', calculate_mean(donations))

**Kết quả:** `477.75`

## 2. Median — Trung vị

Số trung vị là số đứng ở vị trí giữa của dãy đã sắp xếp tăng dần, chia dãy thành 2 nửa bằng nhau:
- độ dài lẻ → số ở giữa là median;
- độ dài chẵn → trung bình của hai số ở giữa.

In [ ]:
def calculate_median(numbers):
    values = sorted(numbers)   # sorted() tạo list mới — không làm thay đổi dữ liệu gốc
    N = len(values)
    mid = N // 2
    if N % 2 == 0:
        return (values[mid - 1] + values[mid]) / 2
    return values[mid]


print('Trung vị số tiền quyên góp là:', calculate_median(donations))
print('Kiểm tra với NumPy:', np.median(donations))

**Kết quả:** `500.0`

> 📝 *So với bản gốc:* bản cũ gọi `numbers.sort()` — **sắp xếp tại chỗ**, làm thay đổi dữ liệu của người gọi hàm (một hiệu ứng phụ nguy hiểm, xem chương 01 bài 01). Bản này dùng `sorted()`.

**Mean vs Median:** median **không bị outlier kéo lệch**. Thêm một ngày quyên góp đột biến 100 000 USD:

In [ ]:
with_outlier = donations + [100_000]
print(f"mean: {np.mean(donations):.1f} → {np.mean(with_outlier):.1f}")
print(f"median: {np.median(donations):.1f} → {np.median(with_outlier):.1f}")

→ Với dữ liệu lệch (thu nhập, giá nhà, số tiền giao dịch), median thường mô tả "giá trị điển hình" tốt hơn. Khi điền missing value (chương 04), dùng median an toàn hơn mean nếu có outlier.

## 3. Mode

Mode là giá trị xuất hiện **nhiều nhất**. Ví dụ điểm thi toán của 20 học sinh (thang 10): điểm 9 xuất hiện nhiều nhất ⇒ `mode = 9`. Nếu nhiều giá trị có cùng tần suất cao nhất, trả về tất cả. Mode dùng được cho cả dữ liệu **phân loại** (category phổ biến nhất).

In [ ]:
points_1 = [7, 8, 9, 2, 10, 9, 9, 9, 9, 4, 5, 6, 1, 5, 6, 7, 8, 6, 1, 10]
points_2 = [7, 8, 9, 2, 10, 9, 9, 9, 9, 4, 5, 6, 1, 5, 6, 7, 8, 6, 1, 10, 6, 6]


def calculate_mode(numbers):
    numbers_freq = Counter(numbers).most_common()   # [(giá trị, số lần), ...] giảm dần theo số lần
    max_count = numbers_freq[0][1]
    return [value for value, count in numbers_freq if count == max_count]


print('Mode của points_1:', calculate_mode(points_1))
print('Mode của points_2:', calculate_mode(points_2))

## 4. Range — Độ trải của dữ liệu

Nếu chỉ quan tâm đến trung bình, ta có thể đánh giá sai dữ liệu. Range = giá trị lớn nhất − giá trị nhỏ nhất. Range rất nhạy với outlier; **IQR** (khoảng tứ phân vị, $Q_3 - Q_1$) là thước đo trải bền vững hơn — và là cơ sở của quy tắc phát hiện outlier $[Q_1 - 1.5\,IQR,\; Q_3 + 1.5\,IQR]$ (chương 04, 22).

In [ ]:
def find_range(numbers):
    lowest, highest = min(numbers), max(numbers)
    print(f'Lowest: {lowest}\tHighest: {highest}\tRange: {highest - lowest}')


find_range(points_2)
q1, q3 = np.percentile(with_outlier, [25, 75])
print(f"Quyên góp có outlier: Q1 = {q1}, Q3 = {q3}, IQR = {q3 - q1}")

## 5. Variance & Standard Deviation — Phương sai và độ lệch chuẩn

Phương sai đo mức độ phân tán của dữ liệu quanh giá trị trung bình; độ lệch chuẩn là căn bậc hai của phương sai (cùng đơn vị với dữ liệu).

<img src="images/Statistical/Population-Variance-Formula.jpg" style="width:35%;height:35%;">

trong đó $\mu$ là mean, $x_i$ là giá trị thứ $i$, $N$ là số lượng dữ liệu.

In [ ]:
points = [7, 8, 9, 2, 10, 9, 9, 9, 9, 4, 5, 6, 1, 5, 6, 7, 8, 6, 1, 10]


def calculate_variance(numbers):
    mean = calculate_mean(numbers)
    squared_diff = [(num - mean) ** 2 for num in numbers]
    return sum(squared_diff) / len(numbers)


print('Phương sai của dãy số là:', calculate_variance(points))
print('Độ lệch chuẩn của dãy số là:', calculate_variance(points) ** 0.5)
print('Kiểm tra với NumPy:', np.var(points), np.std(points))

**Kết quả:** phương sai `7.6475`, độ lệch chuẩn `2.7654`.

### Tổng thể (population) vs mẫu (sample) *(bổ sung)*

Công thức trên chia cho $N$ — đúng khi bạn có **toàn bộ tổng thể**. Trong thực tế ta chỉ có một **mẫu** $n$ phần tử, và ước lượng phương sai tổng thể bằng **phương sai mẫu** chia cho $n - 1$ (hiệu chỉnh Bessel):

$$s^2 = \frac{1}{n-1}\sum_{i=1}^n (x_i - \bar x)^2$$

Vì sao $n - 1$? Vì $\bar x$ được tính từ chính mẫu đó nên các độ lệch $(x_i - \bar x)$ "nhỏ hơn thật" một chút; chia cho $n$ sẽ **ước lượng thấp** phương sai thật. Mô phỏng:

In [ ]:
true_var = 4.0
samples = rng.normal(0, np.sqrt(true_var), size=(100_000, 5))   # 100 000 mẫu, mỗi mẫu n = 5
print(f"Trung bình của var chia n    : {samples.var(axis=1, ddof=0).mean():.3f}  (chệch thấp)")
print(f"Trung bình của var chia n − 1: {samples.var(axis=1, ddof=1).mean():.3f}  (≈ {true_var}, không chệch)")

> ⚠️ `np.var`/`np.std` mặc định `ddof=0` (chia $n$); `pandas.Series.var/std` mặc định `ddof=1`. Hai thư viện cho kết quả khác nhau trên cùng dữ liệu!

## 6. Correlation Coefficient — Hệ số tương quan

Hệ số tương quan Pearson thể hiện mối quan hệ **tuyến tính** giữa hai biến, nằm trong $[-1, 1]$:
- bằng 0: không có tương quan tuyến tính (nhưng **không chắc độc lập**);
- gần 1: quan hệ tuyến tính dương (cùng tăng hoặc cùng giảm);
- gần −1: quan hệ tuyến tính âm.

<img src="images/Statistical/Correlation-Coefficient-Formula.jpg" style="width:40%;height:40%;">

**Tương quan ≠ nhân quả.** Mùa hè, doanh số đồ bơi và doanh số kem cùng tăng mạnh — hệ số tương quan dương gần 1 — nhưng ăn kem không làm người ta mua đồ bơi; cả hai do **thời tiết nóng** gây ra (biến gây nhiễu — *confounder*, chương 38).

Bạn sẽ tự cài đặt công thức này trong **Assignment 91** (tính tương quan giữa 2 ảnh). Ở đây ta xem các bẫy khi đọc hệ số tương quan *(bổ sung)*:

In [ ]:
n = 300
x = rng.uniform(-3, 3, n)
cases = {
    "tuyến tính mạnh": 2 * x + rng.normal(0, 1, n),
    "không liên quan": rng.normal(0, 1, n),
    "phi tuyến (y = x²)": x**2 + rng.normal(0, 0.3, n),
    "1 outlier": np.r_[rng.normal(0, 1, n - 1), 60.0],
}
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, y) in zip(axes, cases.items()):
    xx = x if name != "1 outlier" else np.r_[rng.normal(0, 1, n - 1), 30.0]
    r = np.corrcoef(xx, y)[0, 1]
    rho = stats.spearmanr(xx, y).statistic
    ax.scatter(xx, y, s=8, alpha=0.6)
    ax.set_title(f"{name}\nPearson r = {r:.2f} | Spearman ρ = {rho:.2f}")
fig.tight_layout()
plt.show()

- **Phi tuyến:** $y = x^2$ phụ thuộc hoàn toàn vào $x$ nhưng $r \approx 0$ → Pearson chỉ đo quan hệ **tuyến tính**. *Mutual information* (chương 18) bắt được quan hệ phi tuyến.
- **Outlier:** một điểm duy nhất tạo ra tương quan "mạnh" giả. **Spearman** (tương quan trên thứ hạng) bền vững hơn.
- **Luôn vẽ scatter plot** trước khi tin vào một con số tương quan.

Với nhiều biến, ta dùng **ma trận tương quan** — công cụ EDA rất phổ biến (chương 03):

In [ ]:
from ml_course.datasets import make_house_prices

houses = make_house_prices(n=800)
corr = houses.select_dtypes("number").corr().round(2)
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, corr.iloc[i, j], ha="center", va="center", fontsize=8)
fig.colorbar(im)
ax.grid(False)
ax.set_title("Ma trận tương quan — dữ liệu giá nhà")
plt.show()

## 7. Lấy mẫu và sai lệch lấy mẫu (sampling bias) *(bổ sung)*

Mọi kết luận thống kê và mọi model ML đều giả định **dữ liệu train đại diện cho dữ liệu sẽ gặp khi triển khai**. Khi mẫu bị lệch, model học sai thực tế:

- *Survivorship bias*: chỉ phân tích các startup còn sống → kết luận sai về yếu tố thành công.
- *Selection bias*: model chấm điểm tín dụng chỉ học từ người **đã được** duyệt vay (không biết người bị từ chối có trả nợ không).
- Khảo sát online bỏ sót người không dùng internet.

In [ ]:
population = rng.lognormal(mean=np.log(15), sigma=0.6, size=1_000_000)      # thu nhập (triệu VND)
random_sample = rng.choice(population, 1000, replace=False)
online_sample = rng.choice(population[population > 12], 1000, replace=False)  # chỉ người thu nhập > 12 trả lời
print(f"Thu nhập trung bình thật     : {population.mean():.2f}")
print(f"Mẫu ngẫu nhiên               : {random_sample.mean():.2f}")
print(f"Mẫu lệch (khảo sát online)   : {online_sample.mean():.2f}  ← sai hệ thống, thêm dữ liệu cũng không sửa được")

## 8. Khoảng tin cậy (confidence interval) *(bổ sung)*

Model đạt accuracy 0.85 trên tập test 200 mẫu — con số này **chắc chắn đến đâu**? Khoảng tin cậy 95% cho trung bình (CLT):

$$\bar x \pm t_{0.975,\,n-1}\cdot\frac{s}{\sqrt n}$$

Khi không có công thức (vd: CI cho F1, AUC, median), dùng **bootstrap**: lấy mẫu có hoàn lại từ dữ liệu nhiều lần, tính thống kê trên mỗi mẫu, lấy phân vị 2.5% và 97.5%.

In [ ]:
from ml_course.validation import bootstrap_ci

correct = rng.random(200) < 0.85                  # 1 = model dự đoán đúng mẫu đó
acc = correct.mean()
se = correct.std(ddof=1) / np.sqrt(len(correct))
t = stats.t.ppf(0.975, df=len(correct) - 1)
print(f"Accuracy = {acc:.3f}")
print(f"CI 95% (công thức t): [{acc - t * se:.3f}, {acc + t * se:.3f}]")
est, lo, hi = bootstrap_ci(correct.astype(float), np.mean, n_boot=5000)
print(f"CI 95% (bootstrap)  : [{lo:.3f}, {hi:.3f}]")

Với 200 mẫu test, accuracy 0.85 thực ra có thể là bất cứ đâu trong khoảng ~0.80–0.90. Hai model chênh nhau 1–2% trên tập test nhỏ **chưa chắc** đã khác nhau thật!

## 9. Kiểm định giả thuyết (hypothesis testing) *(bổ sung)*

Câu hỏi: model B có **thực sự** tốt hơn model A, hay chênh lệch chỉ do ngẫu nhiên?

1. **Giả thuyết không** $H_0$: không có khác biệt.
2. Tính **p-value**: xác suất thấy chênh lệch lớn ít nhất như quan sát **nếu $H_0$ đúng**.
3. p-value nhỏ (thường < 0.05) → bác bỏ $H_0$.

Ví dụ: điểm của 2 model trên 10 fold cross-validation (cùng các fold → dùng **paired test**). Ta làm **permutation test** from scratch — trực quan và ít giả định:

In [ ]:
scores_a = np.array([0.81, 0.79, 0.84, 0.80, 0.82, 0.78, 0.83, 0.80, 0.81, 0.82])
scores_b = scores_a + rng.normal(0.012, 0.01, 10)      # B nhỉnh hơn một chút
diff = scores_b - scores_a
observed = diff.mean()

# Nếu H0 đúng, dấu của mỗi chênh lệch là ngẫu nhiên → đảo dấu ngẫu nhiên nhiều lần
signs = rng.choice([-1, 1], size=(20_000, len(diff)))
null_means = (signs * diff).mean(axis=1)
p_perm = np.mean(np.abs(null_means) >= abs(observed))
p_t = stats.ttest_rel(scores_b, scores_a).pvalue
print(f"Chênh lệch trung bình = {observed:.4f}")
print(f"p-value (permutation) = {p_perm:.4f} | p-value (paired t-test) = {p_t:.4f}")

fig, ax = plt.subplots()
ax.hist(null_means, bins=60, alpha=0.7, label="phân phối chênh lệch nếu H0 đúng")
ax.axvline(observed, color="red", lw=2, label="chênh lệch quan sát được")
ax.legend()
plt.show()

**Cẩn thận:**
- p-value **không** phải xác suất $H_0$ đúng.
- "Có ý nghĩa thống kê" ≠ "có ý nghĩa thực tế" — chênh 0.1% accuracy với triệu mẫu có thể rất "significant" nhưng vô nghĩa về kinh doanh.
- **Multiple comparisons:** thử 20 cấu hình model, ngẫu nhiên sẽ có ~1 cấu hình "significant" ở mức 0.05 → đây là lý do **không được tune trên tập test** (chương 15).

## 🧠 Kiểm tra nhanh

1. Dữ liệu thu nhập rất lệch phải. Nên báo cáo mean hay median? Vì sao?
2. `np.std(x)` và `pd.Series(x).std()` cho kết quả khác nhau — vì sao?
3. Pearson r = 0.02 giữa hai biến. Có thể kết luận chúng độc lập không?
4. Model A: 0.861, model B: 0.867 trên test 300 mẫu. Có thể khẳng định B tốt hơn không?

## Tổng kết

Sau bài này, các bạn nên nhớ:
- Mean, Median, Mode, Range/IQR, Variance, Standard Deviation — và khi nào dùng thước đo nào.
- Tổng thể vs mẫu; hiệu chỉnh $n - 1$; `ddof` trong NumPy vs pandas.
- Correlation Coefficient: chỉ đo quan hệ tuyến tính, nhạy với outlier, không phải nhân quả.
- Sampling bias, khoảng tin cậy, bootstrap, kiểm định giả thuyết — nền tảng để **đánh giá model đáng tin cậy** (chương 14, 15).

---
<sub>Bài gốc: *Số liệu thống kê cơ bản* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 1 (MIT License). Đã sửa: hàm median làm thay đổi dữ liệu đầu vào, vòng lặp variance; bổ sung: outlier, IQR, ddof, bẫy tương quan, ma trận tương quan, sampling bias, khoảng tin cậy, bootstrap, kiểm định giả thuyết.</sub>